In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import sqlite3

In [2]:
url = "https://books.toscrape.com/"

response = requests.get(url)
print(response.status_code)

response.status_code

200


200

Reached the website successfully.

In [3]:
soup = BeautifulSoup(response.text, "html.parser")

print(soup.title)

<title>
    All products | Books to Scrape - Sandbox
</title>


Beautiful Soup book is available

In [4]:
book = soup.find("article", class_="product_pod")

print(book)

<article class="product_pod">
<div class="image_container">
<a href="catalogue/a-light-in-the-attic_1000/index.html"><img alt="A Light in the Attic" class="thumbnail" src="media/cache/2c/da/2cdad67c44b002e7ead0cc35693c0e8b.jpg"/></a>
</div>
<p class="star-rating Three">
<i class="icon-star"></i>
<i class="icon-star"></i>
<i class="icon-star"></i>
<i class="icon-star"></i>
<i class="icon-star"></i>
</p>
<h3><a href="catalogue/a-light-in-the-attic_1000/index.html" title="A Light in the Attic">A Light in the ...</a></h3>
<div class="product_price">
<p class="price_color">Â£51.77</p>
<p class="instock availability">
<i class="icon-ok"></i>
    
        In stock
    
</p>
<form>
<button class="btn btn-primary btn-block" data-loading-text="Adding..." type="submit">Add to basket</button>
</form>
</div>
</article>


#Extracting Title

In [5]:
title = book.h3.a["title"]

print(title)

A Light in the Attic


In [6]:
price = book.find("p", class_="price_color").text

print(price)

Â£51.77


#Basic Scraping Logic

In [7]:
title = book.h3.a["title"]

price = book.find("p", class_="price_color").text

rating = book.find("p", class_="star-rating")["class"][1]

availability = book.find("p", class_="instock availability").text.strip()

print("Title:", title)
print("Price:", price)
print("Rating:", rating)
print("Availability:", availability)

Title: A Light in the Attic
Price: Â£51.77
Rating: Three
Availability: In stock


In [8]:
books = soup.find_all("article", class_="product_pod")

print("Number of books:", len(books))

Number of books: 20


#Now Collect first 5 paginated listing pages of the “All products” catalogue

In [9]:
all_books = []

for page in range(1, 6):

    url = f"https://books.toscrape.com/catalogue/page-{page}.html"

    response = requests.get(url)
    response.encoding = "utf-8"

    soup = BeautifulSoup(response.text, "html.parser")

    books = soup.find_all("article", class_="product_pod")

    for book in books:

        title = book.h3.a["title"]

        price = book.find("p", class_="price_color").text

        rating = book.find("p", class_="star-rating")["class"][1]

        availability = book.find(
            "p", class_="instock availability"
        ).text.strip()

        # Get individual book page
        book_link = "https://books.toscrape.com/catalogue/" + book.h3.a["href"]

        book_response = requests.get(book_link)
        book_response.encoding = "utf-8"

        book_soup = BeautifulSoup(book_response.text, "html.parser")

        # Get category from individual book page
        category = book_soup.find(
            "ul", class_="breadcrumb"
        ).find_all("a")[2].text

        all_books.append({
            "title": title,
            "price": price,
            "star_rating": rating,
            "availability": availability,
            "category": category
        })

print("Total books collected:", len(all_books))

Total books collected: 100


In [10]:
df = pd.DataFrame(all_books)

df.head()

,title,price,star_rating,availability,category
0,A Light in the Attic,£51.77,Three,In stock,Poetry
1,Tipping the Velvet,£53.74,One,In stock,Historical Fiction
2,Soumission,£50.10,One,In stock,Fiction
3,Sharp Objects,£47.82,Four,In stock,Mystery
4,Sapiens: A Brief History of Humankind,£54.23,Five,In stock,History


In [11]:
df.shape

(100, 5)

In [12]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   title         100 non-null    object
 1   price         100 non-null    object
 2   star_rating   100 non-null    object
 3   availability  100 non-null    object
 4   category      100 non-null    object
dtypes: object(5)
memory usage: 4.0+ KB


In [13]:
title = book.h3.a["title"]
book_link = "https://books.toscrape.com/catalogue/" + book.h3.a["href"]

book_response = requests.get(book_link)
book_response.encoding = "utf-8"

book_soup = BeautifulSoup(book_response.text, "html.parser")

category = book_soup.find("ul", class_="breadcrumb").find_all("a")[2].text

all_books.append({
    "title": title,
    "price": price,
    "star_rating": rating,
    "availability": availability,
    "category": category
})

In [14]:
df = pd.DataFrame(all_books)
df

,title,price,star_rating,availability,category
0,A Light in the Attic,£51.77,Three,In stock,Poetry
1,Tipping the Velvet,£53.74,One,In stock,Historical Fiction
2,Soumission,£50.10,One,In stock,Fiction
3,Sharp Objects,£47.82,Four,In stock,Mystery
4,Sapiens: A Brief History of Humankind,£54.23,Five,In stock,History
...,...,...,...,...,...
96,"Layered: Baking, Building, and Styling Spectac...",£40.11,One,In stock,Food and Drink
97,Judo: Seven Steps to Black Belt (an Introducto...,£53.90,Two,In stock,Add a comment
98,Join,£35.67,Five,In stock,Science Fiction
99,In the Country We Love: My Family Divided,£22.00,Four,In stock,Nonfiction


In [15]:
df.shape

(101, 5)

In [16]:
df.head()

,title,price,star_rating,availability,category
0,A Light in the Attic,£51.77,Three,In stock,Poetry
1,Tipping the Velvet,£53.74,One,In stock,Historical Fiction
2,Soumission,£50.10,One,In stock,Fiction
3,Sharp Objects,£47.82,Four,In stock,Mystery
4,Sapiens: A Brief History of Humankind,£54.23,Five,In stock,History


In [17]:
df.tail()

,title,price,star_rating,availability,category
96,"Layered: Baking, Building, and Styling Spectac...",£40.11,One,In stock,Food and Drink
97,Judo: Seven Steps to Black Belt (an Introducto...,£53.90,Two,In stock,Add a comment
98,Join,£35.67,Five,In stock,Science Fiction
99,In the Country We Love: My Family Divided,£22.00,Four,In stock,Nonfiction
100,In the Country We Love: My Family Divided,£22.00,Four,In stock,Nonfiction


In [18]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 101 entries, 0 to 100
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   title         101 non-null    object
 1   price         101 non-null    object
 2   star_rating   101 non-null    object
 3   availability  101 non-null    object
 4   category      101 non-null    object
dtypes: object(5)
memory usage: 4.1+ KB


In [19]:
df.describe()

,title,price,star_rating,availability,category
count,101,101,101,101,101
unique,100,99,5,1,29
top,In the Country We Love: My Family Divided,£44.18,Three,In stock,Sequential Art
freq,2,2,22,101,14


In [20]:
df.columns

Index(['title', 'price', 'star_rating', 'availability', 'category'], dtype='object')

In [21]:
df.duplicated().sum()

np.int64(1)

In [22]:
df = df.drop_duplicates().copy()
df.duplicated().sum()

np.int64(0)

In [23]:
df.shape

(100, 5)

In [24]:
df.isna().sum()

,0
title,0
price,0
star_rating,0
availability,0
category,0


In [25]:
{
    "title": title,
    "price": price,
    "star_rating": rating,
    "availability": availability
}

{'title': 'In the Country We Love: My Family Divided',
 'price': '£22.00',
 'star_rating': 'Four',
 'availability': 'In stock'}

In [26]:
df[df["price"].str.strip() == ""]

,title,price,star_rating,availability,category


In [27]:
df["price"].unique()

array(['£51.77', '£53.74', '£50.10', '£47.82', '£54.23', '£22.65',
       '£33.34', '£17.93', '£22.60', '£52.15', '£13.99', '£20.66',
       '£17.46', '£52.29', '£35.02', '£57.25', '£23.88', '£37.59',
       '£51.33', '£45.17', '£12.84', '£37.32', '£30.52', '£25.27',
       '£34.53', '£54.64', '£22.50', '£53.13', '£40.30', '£44.18',
       '£17.66', '£31.05', '£23.82', '£36.89', '£15.94', '£33.29',
       '£18.02', '£19.63', '£52.22', '£33.63', '£57.31', '£26.41',
       '£47.61', '£23.11', '£45.07', '£31.77', '£50.27', '£14.27',
       '£18.78', '£25.52', '£16.28', '£31.12', '£19.49', '£17.27',
       '£19.09', '£56.13', '£56.41', '£56.50', '£45.22', '£38.16',
       '£54.11', '£42.96', '£23.89', '£16.77', '£20.59', '£37.13',
       '£56.06', '£58.11', '£49.05', '£40.76', '£19.73', '£32.24',
       '£41.83', '£39.58', '£39.25', '£25.02', '£51.04', '£19.83',
       '£50.40', '£13.61', '£13.34', '£18.97', '£36.28', '£10.16',
       '£15.44', '£48.41', '£46.35', '£14.07', '£14.86', '£33.

In [28]:
df["price"].isna().sum()

np.int64(0)

In [29]:
df["price_gbp"] = (
    df["price"]
    .str.strip()
    .str.replace("£", "", regex=False)
)

df["price_gbp"] = pd.to_numeric(df["price_gbp"], errors="coerce")

In [30]:
df["price_gbp"] = (
    df["price"]
    .str.strip()
    .str.replace("£", "", regex=False)
)

df["price_gbp"] = pd.to_numeric(
    df["price_gbp"],
    errors="coerce"
)

In [31]:
df["price_gbp"].isna().sum()

np.int64(0)

In [32]:
df[["price", "price_gbp"]].head()

,price,price_gbp
0,£51.77,51.77
1,£53.74,53.74
2,£50.10,50.10
3,£47.82,47.82
4,£54.23,54.23


In [33]:
df["price_gbp"].dtype

dtype('float64')

In [34]:
df["star_rating"].value_counts()

,count
star_rating,
Three,22
One,22
Five,19
Two,19
Four,18


In [35]:
rating_map = {
    "One": 1,
    "Two": 2,
    "Three": 3,
    "Four": 4,
    "Five": 5
}

df["rating"] = df["star_rating"].map(rating_map)

In [36]:
df["rating"].isna().sum()

np.int64(0)

In [37]:
df[["star_rating", "rating"]].head()

,star_rating,rating
0,Three,3
1,One,1
2,One,1
3,Four,4
4,Five,5


In [38]:
df["rating"].dtype

dtype('int64')

In [39]:
df["availability"].value_counts()

,count
availability,
In stock,100


In [40]:
df["in_stock"] = df["availability"].str.strip() == "In stock"
df[["availability", "in_stock"]].head()

,availability,in_stock
0,In stock,True
1,In stock,True
2,In stock,True
3,In stock,True
4,In stock,True


In [41]:
df["in_stock"].dtype

dtype('bool')

In [42]:
GBP_TO_INR = 105.50

df["price_inr"] = df["price_gbp"] * GBP_TO_INR

df[["price_gbp", "price_inr"]].head()

,price_gbp,price_inr
0,51.77,5461.735
1,53.74,5669.570
2,50.10,5285.550
3,47.82,5045.010
4,54.23,5721.265


#Covernsion of currency

In [43]:
GBP_TO_INR = 105.50

df["price_inr"] = df["price_gbp"] * GBP_TO_INR

df["price_inr"] = df["price_inr"].round(2)

df[["price_gbp", "price_inr"]].head()

,price_gbp,price_inr
0,51.77,5461.74
1,53.74,5669.57
2,50.10,5285.55
3,47.82,5045.01
4,54.23,5721.26


In [44]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 100 entries, 0 to 99
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   title         100 non-null    object 
 1   price         100 non-null    object 
 2   star_rating   100 non-null    object 
 3   availability  100 non-null    object 
 4   category      100 non-null    object 
 5   price_gbp     100 non-null    float64
 6   rating        100 non-null    int64  
 7   in_stock      100 non-null    bool   
 8   price_inr     100 non-null    float64
dtypes: bool(1), float64(2), int64(1), object(5)
memory usage: 7.1+ KB


In [45]:
df[["title", "price_gbp", "price_inr", "rating", "in_stock", "category"]].head()

,title,price_gbp,price_inr,rating,in_stock,category
0,A Light in the Attic,51.77,5461.74,3,True,Poetry
1,Tipping the Velvet,53.74,5669.57,1,True,Historical Fiction
2,Soumission,50.10,5285.55,1,True,Fiction
3,Sharp Objects,47.82,5045.01,4,True,Mystery
4,Sapiens: A Brief History of Humankind,54.23,5721.26,5,True,History


In [46]:
conn = sqlite3.connect("zepto_books.db")

cursor = conn.cursor()

In [47]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS categories (
    category_id INTEGER PRIMARY KEY,
    category_name TEXT UNIQUE
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS books (
    book_id INTEGER PRIMARY KEY,
    title TEXT,
    price_gbp REAL,
    price_inr REAL,
    rating INTEGER,
    in_stock INTEGER,
    category_id INTEGER,
    FOREIGN KEY (category_id) REFERENCES categories(category_id)
)
""")

conn.commit()

In [48]:
cursor.execute("""
SELECT name
FROM sqlite_master
WHERE type='table'
""")

cursor.fetchall()

[('categories',), ('books',)]

In [49]:
categories = df["category"].unique()

for category in categories:
    cursor.execute(
        "INSERT OR IGNORE INTO categories (category_name) VALUES (?)",
        (category,)
    )

conn.commit()

In [50]:
cursor.execute("SELECT * FROM categories")

category_data = cursor.fetchall()

category_data

[(1, 'Poetry'),
 (2, 'Historical Fiction'),
 (3, 'Fiction'),
 (4, 'Mystery'),
 (5, 'History'),
 (6, 'Young Adult'),
 (7, 'Business'),
 (8, 'Default'),
 (9, 'Sequential Art'),
 (10, 'Music'),
 (11, 'Science Fiction'),
 (12, 'Politics'),
 (13, 'Travel'),
 (14, 'Thriller'),
 (15, 'Food and Drink'),
 (16, 'Romance'),
 (17, 'Childrens'),
 (18, 'Nonfiction'),
 (19, 'Art'),
 (20, 'Spirituality'),
 (21, 'Philosophy'),
 (22, 'New Adult'),
 (23, 'Contemporary'),
 (24, 'Fantasy'),
 (25, 'Add a comment'),
 (26, 'Science'),
 (27, 'Health'),
 (28, 'Horror'),
 (29, 'Self Help')]

#Catergory Map

In [51]:
cursor.execute("SELECT category_id, category_name FROM categories")

category_map = {
    name: category_id
    for category_id, name in cursor.fetchall()
}

category_map

{'Poetry': 1,
 'Historical Fiction': 2,
 'Fiction': 3,
 'Mystery': 4,
 'History': 5,
 'Young Adult': 6,
 'Business': 7,
 'Default': 8,
 'Sequential Art': 9,
 'Music': 10,
 'Science Fiction': 11,
 'Politics': 12,
 'Travel': 13,
 'Thriller': 14,
 'Food and Drink': 15,
 'Romance': 16,
 'Childrens': 17,
 'Nonfiction': 18,
 'Art': 19,
 'Spirituality': 20,
 'Philosophy': 21,
 'New Adult': 22,
 'Contemporary': 23,
 'Fantasy': 24,
 'Add a comment': 25,
 'Science': 26,
 'Health': 27,
 'Horror': 28,
 'Self Help': 29}

In [52]:
for _, row in df.iterrows():

    category_id = category_map[row["category"]]

    cursor.execute("""
        INSERT INTO books
        (title, price_gbp, price_inr, rating, in_stock, category_id)
        VALUES (?, ?, ?, ?, ?, ?)
    """, (
        row["title"],
        row["price_gbp"],
        row["price_inr"],
        int(row["rating"]),
        int(row["in_stock"]),
        category_id
    ))

conn.commit()

int(row["in_stock"])

1

In [53]:
cursor.execute("SELECT COUNT(*) FROM books")

cursor.fetchone()

(100,)

In [54]:
cursor.execute("SELECT * FROM books LIMIT 5")

cursor.fetchall()

[(1, 'A Light in the Attic', 51.77, 5461.74, 3, 1, 1),
 (2, 'Tipping the Velvet', 53.74, 5669.57, 1, 1, 2),
 (3, 'Soumission', 50.1, 5285.55, 1, 1, 3),
 (4, 'Sharp Objects', 47.82, 5045.01, 4, 1, 4),
 (5, 'Sapiens: A Brief History of Humankind', 54.23, 5721.26, 5, 1, 5)]

In [55]:
cursor.execute("DELETE FROM books")
conn.commit()

cursor.execute("SELECT COUNT(*) FROM books")
cursor.fetchone()

(0,)

In [56]:
for _, row in df.iterrows():

    category_id = category_map[row["category"]]

    cursor.execute("""
        INSERT INTO books
        (title, price_gbp, price_inr, rating, in_stock, category_id)
        VALUES (?, ?, ?, ?, ?, ?)
    """, (
        row["title"],
        row["price_gbp"],
        row["price_inr"],
        int(row["rating"]),
        int(row["in_stock"]),
        category_id
    ))

conn.commit()

cursor.execute("SELECT COUNT(*) FROM books")
cursor.fetchone()

(100,)

In [57]:
cursor.execute("DELETE FROM books")
conn.commit()

cursor.execute("SELECT COUNT(*) FROM books")
cursor.fetchone()

(0,)

In [58]:
for _, row in df.iterrows():

    category_id = category_map[row["category"]]

    cursor.execute("""
        INSERT INTO books
        (title, price_gbp, price_inr, rating, in_stock, category_id)
        VALUES (?, ?, ?, ?, ?, ?)
    """, (
        row["title"],
        row["price_gbp"],
        row["price_inr"],
        int(row["rating"]),
        int(row["in_stock"]),
        category_id
    ))

conn.commit()

cursor.execute("SELECT COUNT(*) FROM books")
cursor.fetchone()

(100,)

In [59]:
cursor.execute("""
SELECT COUNT(*) AS total_books
FROM books
""")

cursor.fetchall()

[(100,)]

In [60]:
cursor.execute("""
SELECT AVG(price_inr) AS average_price
FROM books
""")

cursor.fetchall()

[(3646.1535999999996,)]

In [61]:
cursor.execute("""
SELECT title, rating
FROM books
WHERE rating = 5
""")

cursor.fetchall()

[('Sapiens: A Brief History of Humankind', 5),
 ('Set Me Free', 5),
 ("Scott Pilgrim's Precious Little Life (Scott Pilgrim #1)", 5),
 ('Rip it Up and Start Again', 5),
 ('Chase Me (Paris Nights #2)', 5),
 ('Black Dust', 5),
 ('Worlds Elsewhere: Journeys Around Shakespeare’s Globe', 5),
 ('The Four Agreements: A Practical Guide to Personal Freedom', 5),
 ('The Elephant Tree', 5),
 ("Sophie's World", 5),
 ('Private Paris (Private #10)', 5),
 ('#HigherSelfie: Wake Up Your Life. Free Your Soul. Find Your Tribe.', 5),
 ('We Love You, Charlie Freeman', 5),
 ('Thirst', 5),
 ('The Inefficiency Assassin: Time Management Tactics for Working Smarter, Not Longer',
  5),
 ("The Activist's Tao Te Ching: Ancient Advice for a Modern Revolution", 5),
 ('Princess Jellyfish 2-in-1 Omnibus, Vol. 01 (Princess Jellyfish 2-in-1 Omnibus #1)',
  5),
 ('Princess Between Worlds (Wide-Awake Princess #5)', 5),
 ('Join', 5)]

In [62]:
cursor.execute("""
SELECT c.category_name,
       ROUND(AVG(b.price_inr), 2) AS average_price_inr
FROM books b
JOIN categories c
ON b.category_id = c.category_id
GROUP BY c.category_name
ORDER BY average_price_inr DESC
""")

cursor.fetchall()

[('Historical Fiction', 5669.57),
 ('Politics', 5415.32),
 ('Childrens', 5192.71),
 ('Health', 5174.77),
 ('Self Help', 4889.92),
 ('Travel', 4765.44),
 ('New Adult', 4754.88),
 ('Art', 4660.99),
 ('Fiction', 4628.49),
 ('Music', 4557.25),
 ('Science', 4532.28),
 ('Mystery', 4358.91),
 ('Horror', 4140.88),
 ('Philosophy', 3906.14),
 ('Science Fiction', 3864.47),
 ('Poetry', 3823.17),
 ('Food and Drink', 3680.47),
 ('History', 3580.67),
 ('Business', 3517.37),
 ('Nonfiction', 3426.9),
 ('Sequential Art', 3366.13),
 ('Contemporary', 3351.74),
 ('Add a comment', 3201.08),
 ('Romance', 3154.45),
 ('Thriller', 3125.61),
 ('Fantasy', 2994.62),
 ('Default', 2832.44),
 ('Young Adult', 2642.51),
 ('Spirituality', 2632.23)]

In [63]:
cursor.execute("""
SELECT c.category_name,
       COUNT(b.book_id) AS book_count
FROM categories c
JOIN books b
ON c.category_id = b.category_id
GROUP BY c.category_name
ORDER BY book_count DESC
""")

cursor.fetchall()

[('Sequential Art', 14),
 ('Nonfiction', 12),
 ('Default', 9),
 ('Poetry', 7),
 ('Food and Drink', 5),
 ('Fiction', 5),
 ('Add a comment', 5),
 ('Young Adult', 4),
 ('History', 4),
 ('Fantasy', 4),
 ('Thriller', 3),
 ('Mystery', 3),
 ('Music', 3),
 ('Childrens', 3),
 ('Spirituality', 2),
 ('Science Fiction', 2),
 ('Romance', 2),
 ('Philosophy', 2),
 ('Travel', 1),
 ('Self Help', 1),
 ('Science', 1),
 ('Politics', 1),
 ('New Adult', 1),
 ('Horror', 1),
 ('Historical Fiction', 1),
 ('Health', 1),
 ('Contemporary', 1),
 ('Business', 1),
 ('Art', 1)]

In [64]:
sql_df = pd.read_sql("""
SELECT b.book_id,
       b.title,
       b.price_gbp,
       b.price_inr,
       b.rating,
       b.in_stock,
       c.category_name
FROM books b
JOIN categories c
ON b.category_id = c.category_id
""", conn)

sql_df.head()

,book_id,title,price_gbp,price_inr,rating,in_stock,category_name
0,1,A Light in the Attic,51.77,5461.74,3,1,Poetry
1,2,Tipping the Velvet,53.74,5669.57,1,1,Historical Fiction
2,3,Soumission,50.10,5285.55,1,1,Fiction
3,4,Sharp Objects,47.82,5045.01,4,1,Mystery
4,5,Sapiens: A Brief History of Humankind,54.23,5721.26,5,1,History


In [65]:
books_df = pd.read_sql("SELECT * FROM books", conn)

categories_df = pd.read_sql("SELECT * FROM categories", conn)

merged_df = pd.merge(
    books_df,
    categories_df,
    on="category_id",
    how="left"
)

merged_df.head()

,book_id,title,price_gbp,price_inr,rating,in_stock,category_id,category_name
0,1,A Light in the Attic,51.77,5461.74,3,1,1,Poetry
1,2,Tipping the Velvet,53.74,5669.57,1,1,2,Historical Fiction
2,3,Soumission,50.10,5285.55,1,1,3,Fiction
3,4,Sharp Objects,47.82,5045.01,4,1,4,Mystery
4,5,Sapiens: A Brief History of Humankind,54.23,5721.26,5,1,5,History


In [66]:
merged_df.shape

(100, 8)

In [67]:
merged_df.isna().sum()

,0
book_id,0
title,0
price_gbp,0
price_inr,0
rating,0
in_stock,0
category_id,0
category_name,0


# Final SQL Queries

The following queries shows filtering, sorting, limiting results, distinct values, range filtering and joining normalized tables.

In [68]:
# Query 1 - WHERE
query1 = """
SELECT title, price_inr, rating
FROM books
WHERE rating = 5
"""
display(pd.read_sql(query1, conn))


# Query 2 - ORDER BY and LIMIT
query2 = """
SELECT title, price_inr
FROM books
ORDER BY price_inr DESC
LIMIT 10
"""
display(pd.read_sql(query2, conn))


# Query 3 - DISTINCT
query3 = """
SELECT DISTINCT category_name
FROM categories
ORDER BY category_name
"""
display(pd.read_sql(query3, conn))


# Query 4 - BETWEEN
query4 = """
SELECT title, price_inr
FROM books
WHERE price_inr BETWEEN 2000 AND 4000
ORDER BY price_inr
"""
display(pd.read_sql(query4, conn))


# Query 5 - JOIN
query5 = """
SELECT b.title,
       b.price_inr,
       b.rating,
       c.category_name
FROM books b
JOIN categories c
ON b.category_id = c.category_id
ORDER BY b.rating DESC, b.price_inr DESC
LIMIT 10
"""
sql_join_df = pd.read_sql(query5, conn)

display(sql_join_df)

,title,price_inr,rating
0,Sapiens: A Brief History of Humankind,5721.26,5
1,Set Me Free,1842.03,5
2,Scott Pilgrim's Precious Little Life (Scott Pi...,5516.60,5
3,Rip it Up and Start Again,3694.61,5
4,Chase Me (Paris Nights #2),2665.98,5
5,Black Dust,3642.92,5
6,Worlds Elsewhere: Journeys Around Shakespeare’...,4251.65,5
7,The Four Agreements: A Practical Guide to Pers...,1863.13,5
8,The Elephant Tree,2513.01,5
9,Sophie's World,1681.67,5


,title,price_inr
0,The Death of Humanity: and the Case for Life,6130.60
1,Slow States of Collapse: Poems,6046.20
2,Our Band Could Be Your Life: Scenes from the A...,6039.88
3,The Past Never Ends,5960.75
4,The Pioneer Woman Cooks: Dinnertime: Comfort C...,5951.25
5,Masks and Shadows,5950.20
6,The Secret of Dreadwillow Carse,5921.72
7,The Electric Pencil: Drawings from Inside Stat...,5914.33
8,Birdsong: A Story in Pictures,5764.52
9,Sapiens: A Brief History of Humankind,5721.26


,category_name
0,Add a comment
1,Art
2,Business
3,Childrens
4,Contemporary
5,Default
6,Fantasy
7,Fiction
8,Food and Drink
9,Health


,title,price_inr
0,"Pop Gun War, Volume 1: Gift",2001.33
1,The Torch Is Passed: A Harding Family Story,2014.00
2,This One Summer,2056.19
3,"In a Dark, Dark Wood",2070.96
4,The Age of Genius: The Seventeenth Century and...,2081.52
5,Reskilling America: Learning to Labor in the T...,2092.06
6,Lumberjanes Vol. 3: A Terrible Plan (Lumberjan...,2101.56
7,The Inefficiency Assassin: Time Management Tac...,2172.24
8,Shakespeare's Sonnets,2179.63
9,In the Country We Love: My Family Divided,2321.00


,title,price_inr,rating,category_name
0,Sapiens: A Brief History of Humankind,5721.26,5,History
1,Scott Pilgrim's Precious Little Life (Scott Pi...,5516.60,5,Sequential Art
2,"We Love You, Charlie Freeman",5303.48,5,Fiction
3,Private Paris (Private #10),5022.85,5,Fiction
4,Worlds Elsewhere: Journeys Around Shakespeare’...,4251.65,5,Nonfiction
5,Join,3763.19,5,Science Fiction
6,Rip it Up and Start Again,3694.61,5,Music
7,Black Dust,3642.92,5,Romance
8,The Activist's Tao Te Ching: Ancient Advice fo...,3401.32,5,Spirituality
9,Chase Me (Paris Nights #2),2665.98,5,Romance


#SQL JOIN and Pandas Merge Comparison

The SQL JOIN result is reproduced using `pd.merge()` to verify that both the approaches return equivalent data.

In [69]:
books_df = pd.read_sql("SELECT * FROM books", conn)
categories_df = pd.read_sql("SELECT * FROM categories", conn)

merged_df = pd.merge(
    books_df,
    categories_df,
    on="category_id",
    how="inner"
)

pandas_join_df = (
    merged_df[
        ["title", "price_inr", "rating", "category_name"]
    ]
    .sort_values(
        by=["rating", "price_inr"],
        ascending=[False, False]
    )
    .head(10)
    .reset_index(drop=True)
)

sql_join_compare = sql_join_df.reset_index(drop=True)

print("SQL JOIN Result:")
display(sql_join_compare)

print("Pandas Merge Result:")
display(pandas_join_df)

print(
    "Both results are equivalent:",
    sql_join_compare.equals(pandas_join_df)
)

SQL JOIN Result:


,title,price_inr,rating,category_name
0,Sapiens: A Brief History of Humankind,5721.26,5,History
1,Scott Pilgrim's Precious Little Life (Scott Pi...,5516.60,5,Sequential Art
2,"We Love You, Charlie Freeman",5303.48,5,Fiction
3,Private Paris (Private #10),5022.85,5,Fiction
4,Worlds Elsewhere: Journeys Around Shakespeare’...,4251.65,5,Nonfiction
5,Join,3763.19,5,Science Fiction
6,Rip it Up and Start Again,3694.61,5,Music
7,Black Dust,3642.92,5,Romance
8,The Activist's Tao Te Ching: Ancient Advice fo...,3401.32,5,Spirituality
9,Chase Me (Paris Nights #2),2665.98,5,Romance


Pandas Merge Result:


,title,price_inr,rating,category_name
0,Sapiens: A Brief History of Humankind,5721.26,5,History
1,Scott Pilgrim's Precious Little Life (Scott Pi...,5516.60,5,Sequential Art
2,"We Love You, Charlie Freeman",5303.48,5,Fiction
3,Private Paris (Private #10),5022.85,5,Fiction
4,Worlds Elsewhere: Journeys Around Shakespeare’...,4251.65,5,Nonfiction
5,Join,3763.19,5,Science Fiction
6,Rip it Up and Start Again,3694.61,5,Music
7,Black Dust,3642.92,5,Romance
8,The Activist's Tao Te Ching: Ancient Advice fo...,3401.32,5,Spirituality
9,Chase Me (Paris Nights #2),2665.98,5,Romance


Both results are equivalent: True
